# Private Spark Connect server quickstart

This notebook creates a private Spark Connect server on Kubernetes that pulls its image with an image pull Secret, receives a credential as an environment variable from a Kubernetes Secret, and runs on nodes chosen by a node selector. It previews the server offline, waits for a real query, runs a small query, and deletes the server.

**This notebook contains no credentials, and none are needed to run it.** An administrator creates the Secrets in the namespace beforehand. The notebook only names them; Kubernetes resolves the values when it starts each container, and spark-on-k8s never reads them. Every client of a server can read the server's environment variables through queries, so give a server only credentials that all of its users may hold.

Follow the [private server quickstart](https://hussein.awala.fr/spark-on-k8s/develop/user-guide/jupyter/#private-server-quickstart) to prepare the Secrets, kernel and settings. Keep `notebook_support.py` beside this notebook. Run the cells in order; this notebook has no kernel-restart boundary.

## 1. Check the kernel and settings

Besides the cluster settings of the main Jupyter example, set `SPARK_NOTEBOOK_PULL_SECRET` to the image pull Secret's name and `SPARK_NOTEBOOK_CREDENTIAL` to a `SECRET:KEY` reference, such as `notebook-credentials:api-token`. Optionally set `SPARK_NOTEBOOK_NODE_SELECTOR` to `KEY=VALUE` for your node pool; the default `kubernetes.io/os=linux` matches every Linux node.

The receipt `quickstart-receipt.json` in the artifact directory identifies the server for status and deletion. If it exists, delete that server first, as shown at the end of this notebook.

In [ ]:
from __future__ import annotations

import importlib.metadata as metadata
import json
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))  # Only the notebook directory, including its companion helper.

import spark_on_k8s
from notebook_support import (
    LoopbackTunnel,
    api_for,
    client_for,
    reserve_receipt,
    settings,
    wait_for_cleanup,
    write_json,
)
from spark_on_k8s.client import ExecutorInstances, PodResources
from spark_on_k8s.connect_server import ConnectServerConfig

cfg = settings()
client = client_for(cfg)
references = ("SPARK_NOTEBOOK_PULL_SECRET", "SPARK_NOTEBOOK_CREDENTIAL")
missing = [name for name in references if not os.environ.get(name)]
if missing:
    raise ValueError("Set these Secret references: " + ", ".join(missing))
pull_secret = os.environ["SPARK_NOTEBOOK_PULL_SECRET"]
credential = os.environ["SPARK_NOTEBOOK_CREDENTIAL"]  # A SECRET:KEY reference, never a value.
selector = os.environ.get("SPARK_NOTEBOOK_NODE_SELECTOR", "kubernetes.io/os=linux")
node_label, _, node_value = selector.partition("=")
receipt = cfg.artifact_dir / "quickstart-receipt.json"
assert not receipt.exists(), "A receipt exists: delete that server before creating another."
if os.environ.get("SPARK_NOTEBOOK_EXPECTED_PYTHON"):
    assert Path(sys.executable) == Path(os.environ["SPARK_NOTEBOOK_EXPECTED_PYTHON"])
    assert Path(spark_on_k8s.__file__).resolve().is_relative_to(Path(sys.prefix).resolve())
client_version = metadata.version("pyspark-client")
if client_version.split(".")[:2] != cfg.spark_version.split(".")[:2]:
    raise RuntimeError(
        f"Install pyspark-client=={cfg.spark_version} to match the server; this kernel has {client_version}."
    )
print(
    {
        "python": sys.version.split()[0],
        "pyspark_client": client_version,
        "context": cfg.context,
        "namespace": cfg.namespace,
        "pull_secret": pull_secret,
        "credential_reference": credential,
        "node_selector": {node_label: node_value},
    }
)

## 2. Describe the server and preview it offline

The configuration names the pull Secret, maps the credential's Secret key to the `QUICKSTART_API_TOKEN` variable on the driver and executors, and selects nodes for both. Invalid names, label syntax or references fail here, before any Kubernetes request.

`plan_connect_server` renders the resources that creation would request without contacting the cluster. The driver's variable appears only as a `secretKeyRef`: no Secret is read, so no value can appear.

In [ ]:
server_config = ConnectServerConfig(
    image=cfg.image,
    server_jar=cfg.server_jar,
    namespace=cfg.namespace,
    service_account=cfg.service_account,
    app_name="private-quickstart",
    image_pull_policy=cfg.image_pull_policy,
    image_pull_secrets=[pull_secret],
    driver_env_from_secrets={"QUICKSTART_API_TOKEN": credential},
    executor_env_from_secrets={"QUICKSTART_API_TOKEN": credential},
    driver_node_selector={node_label: node_value},
    executor_node_selector={node_label: node_value},
    driver_resources=PodResources(cpu=1, memory=512, memory_overhead=384),
    executor_resources=PodResources(cpu=1, memory=512, memory_overhead=384),
    executor_instances=ExecutorInstances(initial=1),
    spark_conf={
        "spark.sql.shuffle.partitions": "2",
        "spark.kubernetes.executor.request.cores": "0.25",
    },
)
plan = client.plan_connect_server(server_config).to_dict()
planned_driver = next(item for item in plan["resources"] if item["kind"] == "Pod")["spec"]
references = [env for env in planned_driver["containers"][0]["env"] if "valueFrom" in env]
print(
    {
        "planned_resources": [item["kind"] for item in plan["resources"]],
        "driver_secret_references": [env for env in references if "secretKeyRef" in env["valueFrom"]],
        "driver_node_selector": planned_driver["nodeSelector"],
        "image_pull_secrets": server_config.image_pull_secrets,
    }
)

## 3. Create the server and save its receipt

Creation means that Kubernetes accepted the resources; readiness is checked next. The receipt stores Kubernetes UIDs for safe deletion and is published without overwriting an existing file. If creation or publication fails, a `.pending` marker blocks retries: inspect the failure and the namespace before removing it, because a missing receipt does not prove that no server exists.

In [ ]:
with reserve_receipt(receipt):
    server = client.create_connect_server(server_config)
    try:
        write_json(receipt, {"schema_version": 1, "server": server.to_dict()}, exclusive=True)
    except BaseException:
        print("Server resources exist; retain this full identity for recovery:")
        print(json.dumps({"schema_version": 1, "server": server.to_dict()}, indent=2))
        raise
    print({"app_id": server.run.app_id, "endpoint": server.endpoint, "receipt": receipt.name})

## 4. Open a local tunnel and wait for a real query

Kubernetes resolves the Secret references when it starts each container. A missing pull Secret keeps the image from being pulled, and a missing Secret or key keeps the container from starting; an unmatched node selector leaves the Pod pending. In each case readiness times out and leaves the server running: inspect it with `spark-on-k8s app diagnose`, then delete it as shown at the end of this notebook.

The helper opens a loopback-only tunnel; it needs permission to port-forward to, and exec in, the driver Pod.

In [ ]:
tunnel = LoopbackTunnel(cfg, server, "quickstart")
profile = tunnel.start()
ready = client.wait_connect_server_ready(server, profile=profile, timeout=180)
assert ready["query_ready"] is True
assert ready["probe"]["app_id"] == server.run.app_id
print(
    {
        "query_ready": True,
        "spark_version": ready["spark_version"],
        "client_version": ready["client_version"],
        "warnings": ready["warnings"],
    }
)

## 5. Run a query and check the credential without printing it

The first query sums a distributed range on the executors. The second asks each task only whether `QUICKSTART_API_TOKEN` is set in its executor's environment; it never returns the value. Your own code would pass such a variable to a connector or client library running on the server.

The last part reads the executor Pods from Kubernetes to show where the settings landed: a node selector, the pull Secret's name and a reference to the Secret key.

In [ ]:
from pyspark.sql import functions as F

# One row per task. reflect returns the text 'null' for an unset variable, and Spark cannot
# aggregate this non-deterministic check, so each row returns only a boolean to count here.
present = "reflect('java.lang.System', 'getenv', 'QUICKSTART_API_TOKEN') <> 'null' AS present"

with profile.session() as spark:
    assert spark.conf.get("spark.app.id") == server.run.app_id
    total = spark.range(0, 1000, 1, 4).agg(F.sum("id").alias("total")).first().total
    rows = spark.sql(f"SELECT {present} FROM range(0, 4, 1, 4)").collect()
tasks_with_credential = sum(row.present is True for row in rows)
assert total == 499_500
assert len(rows) == 4 and tasks_with_credential == 4, "QUICKSTART_API_TOKEN is not set on the executors"

with api_for(client) as api:
    pods = api.list_namespaced_pod(
        cfg.namespace, label_selector="spark-role=executor", _request_timeout=(5, 20)
    ).items
executors = []
for pod in pods:
    if not any(owner.uid == server.run.driver_uid for owner in pod.metadata.owner_references or []):
        continue
    container = pod.spec.containers[0]
    reference = next(env for env in container.env if env.name == "QUICKSTART_API_TOKEN").value_from
    executors.append(
        {
            "name": pod.metadata.name,
            "uid": pod.metadata.uid,
            "node_selector": pod.spec.node_selector,
            "image_pull_secrets": [secret.name for secret in pod.spec.image_pull_secrets or []],
            "credential_reference": f"{reference.secret_key_ref.name}:{reference.secret_key_ref.key}",
        }
    )
assert executors
assert all(item["node_selector"] == {node_label: node_value} for item in executors)
assert all(item["image_pull_secrets"] == [pull_secret] for item in executors)
assert all(item["credential_reference"] == credential for item in executors)
print(
    {
        "range_sum": total,
        "credential_set_in_executor_tasks": True,
        "executors": [{k: v for k, v in item.items() if k != "uid"} for item in executors],
    }
)

## 6. Delete the server and observe cleanup

Closing a session or the tunnel does not stop the server. This cell requests UID-checked deletion of the recorded driver and Services and waits for the owned executors and executor resources to disappear. Repeating the deletion reports the server as absent.

In [ ]:
tunnel.close()
deletion = client.delete_connect_server(server)
cleanup = wait_for_cleanup(client, server, [item["uid"] for item in executors])
absent = client.delete_connect_server(server)
assert absent["status"] == "absent"
report = {
    "status": "passed",
    "spark_version": ready["spark_version"],
    "client_version": client_version,
    "app_id": server.run.app_id,
    "driver_uid": server.run.driver_uid,
    "range_sum": total,
    "credential_set_in_executor_tasks": tasks_with_credential == 4,
    "pull_secret": pull_secret,
    "credential_reference": credential,
    "node_selector": {node_label: node_value},
    "executors": [{k: v for k, v in item.items() if k != "uid"} for item in executors],
    "deletion": deletion,
    "cleanup": cleanup,
    "repeated_delete": absent,
    "tunnels_closed": tunnel.process.poll() is not None,
}
write_json(cfg.artifact_dir / "quickstart-result.json", report)
print(
    {
        "status": report["status"],
        "owned_resources_remaining": cleanup["owned_resources_remaining"],
        "repeated_delete": absent["status"],
        "tunnel_closed": report["tunnels_closed"],
    }
)

## If the notebook stops early

The server keeps running until it is deleted. The receipt is enough to inspect or delete it from a terminal, using the CLI from the same environment and the artifact directory configured in the guide:

    spark-on-k8s connect server status "$SPARK_NOTEBOOK_ARTIFACT_DIR/quickstart-receipt.json" --kubeconfig "$SPARK_NOTEBOOK_KUBECONFIG" --context "$SPARK_NOTEBOOK_CONTEXT" --output json
    spark-on-k8s connect server delete "$SPARK_NOTEBOOK_ARTIFACT_DIR/quickstart-receipt.json" --kubeconfig "$SPARK_NOTEBOOK_KUBECONFIG" --context "$SPARK_NOTEBOOK_CONTEXT" --output json

Deletion requests cleanup; keep the receipt until the resources are gone. A notebook-owned tunnel can outlive an abrupt kernel exit; stop it when you no longer need it.

## References

- [Private images, credentials and node placement](https://hussein.awala.fr/spark-on-k8s/develop/user-guide/spark_connect_server/#private-images-credentials-and-node-placement)
- [Build an ML image for Spark Connect](https://hussein.awala.fr/spark-on-k8s/develop/user-guide/connect_ml_image/)
- [Kubernetes Secrets as environment variables](https://kubernetes.io/docs/concepts/configuration/secret/#using-secrets-as-environment-variables)
- [Pulling images from a private registry](https://kubernetes.io/docs/tasks/configure-pod-container/pull-image-private-registry/)